<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Exercise: Build Your First Tool Agent </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

## Purpose
Implement a basic agent.

---

## Learning Goals
- Understand how agents select tools
- Implement a simple decision loop
- Produce a reasoning trace

## Scenario

You are building an AI assistant that helps answer business questions.

The assistant should:
1. Retrieve company data
2. Perform calculations
3. Return a final answer

---

## Example Query
"What is the revenue growth from 2022 to 2023?"

---

## Important
A fixed pipeline cannot handle this well.
Your job is to build an **agent that decides what to do next**.

## Requirements

Your agent MUST:

- Use at least **2 tools**
- Implement **tool selection logic**
- Run in a **loop (multi-step execution)**
- Print a **reasoning trace**

---

## Constraints
- Maximum steps = 5
- Allowed tools:
  - retrieve_docs
  - calculator

---

## Rules
- Do NOT hardcode final answers
- The agent must decide actions dynamically

In [ ]:
import ast
import operator as op
from dataclasses import dataclass
from typing import Callable, Dict, Any

# Safe eval (same as demo)
_operators = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
}

def safe_eval(expr: str) -> float:
    node = ast.parse(expr, mode='eval')

    def _eval(n):
        if isinstance(n, ast.Expression):
            return _eval(n.body)
        if isinstance(n, ast.Constant):
            return n.value
        if isinstance(n, ast.BinOp):
            return _operators[type(n.op)](_eval(n.left), _eval(n.right))
        raise ValueError("Unsupported")

    return _eval(node)

In [ ]:
@dataclass
class Tool:
    name: str
    func: Callable[[str], Dict[str, Any]]
    description: str = ""
    returns_final: bool = False


class ToolRegistry:
    def __init__(self):
        self._tools = {}

    def register(self, tool: Tool):
        self._tools[tool.name] = tool

    def get(self, name: str):
        return self._tools[name]


registry = ToolRegistry()

In [ ]:
# -------------------------------
# Tools (DO NOT MODIFY)
# -------------------------------

def retrieve_tool(query: str):
    return {
        "output": {"revenue_2023": 100, "revenue_2022": 80},
        "tool": "retrieve",
        "final": False
    }


def calculator_tool(query: str):
    try:
        # Safely convert string to dictionary
        if isinstance(query, str):
            data = ast.literal_eval(query)
        else:
            data = query

        # Extract values
        rev_2023 = data["revenue_2023"]
        rev_2022 = data["revenue_2022"]

        # Compute growth
        growth = rev_2023 - rev_2022

        return {"output": growth, "tool": "calculator", "final": False}

    except Exception as e:
        return {"output": f"error: {e}", "tool": "calculator", "final": False}


def answer_tool(query: str):
    return {"output": f"Final answer: {query}", "tool": "answer", "final": True}


registry.register(Tool("retrieve", retrieve_tool))
registry.register(Tool("calculator", calculator_tool))
registry.register(Tool("answer", answer_tool, returns_final=True))

In [ ]:
# -------------------------------
# TODO: Implement Tool Selection
# -------------------------------

def choose_tool(state):
    if isinstance(state, str):
        return "retrieve", "need data"

    if isinstance(state, dict):
        return "calculator", "have data, compute growth"

    return "answer", "final step"

In [ ]:
def run_agent(query, max_steps=5):
    state = query

    for step in range(max_steps):
        print(f"\nStep {step+1}")

        tool_name, reason = choose_tool(state)
        print("Tool:", tool_name, "| Reason:", reason)

        tool = registry.get(tool_name)

        result = tool.func(str(state))
        print("Output:", result["output"])

        if result["final"]:
            return result

        state = result["output"]

    return {"output": "Max steps reached"}

In [ ]:
run_agent("What is the revenue growth from 2022 to 2023?")


Step 1
Tool: retrieve | Reason: need data
Output: {'revenue_2023': 100, 'revenue_2022': 80}

Step 2
Tool: calculator | Reason: have data, compute growth
Output: 20

Step 3
Tool: answer | Reason: final step
Output: Final answer: 20


{'output': 'Final answer: 20', 'tool': 'answer', 'final': True}

## Expected Behavior

Your agent should:

Step 1:
- Decide to retrieve data

Step 2:
- Decide to calculate growth

Step 3:
- Return final answer

---

## Expected Trace Format

===== Step 1 =====  
Decision: retrieve_docs  
Observation: {...}

===== Step 2 =====  
Decision: calculator  
Observation: 20

===== Step 3 =====  
Decision: final_answer  

Final Output: Revenue growth is 20

## Reflection Questions

1. What makes your system an **agent** instead of a pipeline?

2. Where is the **decision-making logic** implemented?

3. What would happen if:
   - The tool returned incorrect data?
   - The calculator failed?

4. How would you improve:
   - Tool selection?
   - Flexibility of the agent?

5. What limitations does your current agent have?